In [1]:
import chromadb

In [2]:
# 创建客户端
client = chromadb.Client()

In [3]:
# 创建集合
collection1 = client.get_or_create_collection(name="collection1")
collection2 = client.get_or_create_collection(name="collection2")

## 新增数据（C-create）

In [4]:
# 1. 直接添加文本数据
collection1.add(
    ids=["1", "2", "3"],
    documents=[
        "This is the first document",
        "This is the second document",
        "This is the third document",
    ],
    metadatas=[
        {'chapter': 3, 'verse': 16},
        {'chapter': 3, 'verse': 5},
        {'chapter': 15, 'verse': 12}
    ]
)

In [32]:
collection1.add(
    ids=["4", "5", "6"],
    documents=[
        "Hello world!",
        "Goodbye World!",
        "Hello Chroma!",
    ],
    metadatas=[
        {'chapter': 4, 'verse': 2},
        {'chapter': 8, 'verse': 7},
        {'chapter': 11, 'verse': 10}
    ]
)

In [33]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [-0.06131598,  0.08817799,  0.02569241, ...,  0.06425828,
          0.09380952,  0.04365632],
        [-0.05396886,  0.04353058, -0.02860478, ...,  0.02393955,
          0.09608028,  0.05749847],
        [-0.02038684,  0.02528079, -0.0005662 , ..., -0.03065239,
          0.03755689,  0.03742788],
        [ 0.0361899 ,  0.0748386 ,  0.02247373, ..., -0.02649769,
         -0.02245254, -0.02141618],
        [-0.10963725,  0.03049186,  0.01641894, ...,  0.00313871,
          0.05246637, -0.02596473]], shape=(6, 384)),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world!',
  'Goodbye World!',
  'Hello Chroma!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'verse': 5, 'cha

In [6]:
collection1.peek()['embeddings'].shape

(3, 384)

In [7]:
# 2. 添加数据的同时，指定嵌入向量
collection2.add(
    ids=["1", "2", "3"],
    documents=[
        "This is the first document",
        "This is the second document",
        "This is the third document",
    ],
    embeddings=[
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]
    ]
)

In [8]:
collection2.peek()['embeddings'].shape

(3, 3)

In [9]:
# collection2.add(
#     ids="4",
#     documents=[
#         "This is the forth document",
#     ],
#
# ) # 报错

## 查询数据（R-read）

In [11]:
# 1. 直接获取数据
collection1.get(ids=["2", "3"], include=["metadatas", "documents", "embeddings"])

{'ids': ['2', '3'],
 'embeddings': array([[-6.13159798e-02,  8.81779939e-02,  2.56924052e-02,
          2.88427882e-02,  2.98360977e-02,  1.40053481e-02,
         -3.17783356e-02,  1.60764940e-02,  2.54136920e-02,
          7.86885736e-04,  4.16738205e-02,  7.78555498e-02,
         -4.99317050e-03, -3.92468013e-02, -5.77454679e-02,
          1.92843135e-02, -4.96806502e-02, -6.30019046e-03,
          3.50779295e-02,  5.36748394e-02,  5.07013090e-02,
          6.13464490e-02,  3.10037062e-02, -3.32533121e-02,
          1.34151401e-02,  6.73581883e-02, -1.07689939e-01,
          5.52610233e-02,  4.11786214e-02, -8.33426937e-02,
          2.40026582e-02,  6.34173378e-02, -8.38166289e-03,
          8.88874941e-03,  3.83739844e-02, -5.77810630e-02,
          8.14786702e-02,  2.19539665e-02,  2.92267036e-02,
          2.35394165e-02,  1.69644952e-02, -1.05684824e-01,
         -4.40488607e-02, -1.46724917e-02, -1.13242608e-03,
          3.50902379e-02, -5.47159240e-02,  7.12127285e-03,
      

In [26]:
# 2. 增加筛选条件
collection1.get(
    where={"chapter": {"$eq": 3}},
    where_document={'$contains': 'second'},
)

{'ids': ['2'],
 'embeddings': None,
 'documents': ['This is the second document'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 5}]}

In [36]:
# 3. 相似查询
# 3.1 直接传入数据（文本）
collection1.query(
    query_texts="hello",
    where={"chapter": {"$gte": 4}},
    n_results=3
)

{'ids': [['4', '6', '5']],
 'embeddings': None,
 'documents': [['Hello world!', 'Hello Chroma!', 'Goodbye World!']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'verse': 2, 'chapter': 4},
   {'chapter': 11, 'verse': 10},
   {'chapter': 8, 'verse': 7}]],
 'distances': [[0.5710346102714539, 0.8545363545417786, 1.330894947052002]]}

In [39]:
# 3.2 传入向量
collection2.query(
    query_embeddings=[1.1, 2.3, 3.4],
    include=["metadatas", "documents", "embeddings", "distances"],
)

{'ids': [['1', '2', '3']],
 'embeddings': [array([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]])],
 'documents': [['This is the first document',
   'This is the second document',
   'This is the third document']],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.2600000500679016, 22.459999084472656, 98.66000366210938]]}

## 更新操作（U-update）

In [40]:
# 插入相同id的数据，不会改变
collection1.add(
    ids=["4", "5", "6"],
    documents=[
        "Hello world 111!",
        "Goodbye World 111!",
        "Hello Chroma 111!",
    ],
    metadatas=[
        {'chapter': 4, 'verse': 2},
        {'chapter': 8, 'verse': 7},
        {'chapter': 11, 'verse': 10}
    ]
)

In [41]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [-0.06131598,  0.08817799,  0.02569241, ...,  0.06425828,
          0.09380952,  0.04365632],
        [-0.05396886,  0.04353058, -0.02860478, ...,  0.02393955,
          0.09608028,  0.05749847],
        [-0.02038684,  0.02528079, -0.0005662 , ..., -0.03065239,
          0.03755689,  0.03742788],
        [ 0.0361899 ,  0.0748386 ,  0.02247373, ..., -0.02649769,
         -0.02245254, -0.02141618],
        [-0.10963725,  0.03049186,  0.01641894, ...,  0.00313871,
          0.05246637, -0.02596473]], shape=(6, 384)),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world!',
  'Goodbye World!',
  'Hello Chroma!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'verse': 16, 'chapter': 3},
  {'verse': 5, 'cha

In [42]:
# update更改指定id的数据
collection1.update(
    ids=["4", "5", "6"],
    documents=[
        "Hello world 111!",
        "Goodbye World 111!",
        "Hello Chroma 111!",
    ],
    metadatas=[
        {'chapter': 4, 'verse': 2},
        {'chapter': 5, 'verse': 7},
        {'chapter': 11, 'verse': 10}
    ]
)

In [43]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [-0.06131598,  0.08817799,  0.02569241, ...,  0.06425828,
          0.09380952,  0.04365632],
        [-0.05396886,  0.04353058, -0.02860478, ...,  0.02393955,
          0.09608028,  0.05749847],
        [-0.08065046,  0.03005748, -0.00960963, ...,  0.00635423,
         -0.06554208, -0.05310866],
        [-0.01924923,  0.08375515,  0.00162326, ..., -0.01317479,
         -0.08101653, -0.07411239],
        [-0.15383205,  0.03568971, -0.01151463, ...,  0.02735919,
         -0.01315351, -0.06161029]], shape=(6, 384)),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world 111!',
  'Goodbye World 111!',
  'Hello Chroma 111!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'ver

In [48]:
# upsert = update + insert
collection1.upsert(
    ids=["4", "5", "6", "7"],
    documents=[
        "Hello world 222!",
        "Goodbye World 222!",
        "Hello Chroma 222!",
        "Hello Chroma 22222!",
    ],
    metadatas=[
        {'chapter': 4, 'verse': 2},
        {'chapter': 5, 'verse': 7},
        {'chapter': 12, 'verse': 13},
        {'chapter': 9, 'verse': 6, "price": 50.0}
    ]
)

In [49]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6', '7'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [-0.06131598,  0.08817799,  0.02569241, ...,  0.06425828,
          0.09380952,  0.04365632],
        [-0.05396886,  0.04353058, -0.02860478, ...,  0.02393955,
          0.09608028,  0.05749847],
        ...,
        [ 0.00859756,  0.037237  , -0.00891263, ..., -0.06948347,
         -0.10250813, -0.0632599 ],
        [-0.13393995, -0.00590101, -0.01544957, ...,  0.01599919,
         -0.01226634, -0.05547559],
        [-0.1351217 , -0.0036003 , -0.00462356, ...,  0.01359502,
          0.00168626, -0.05872585]], shape=(7, 384)),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world 222!',
  'Goodbye World 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metada

## 删除数据（D-delete）

In [51]:
collection1.delete(
    ids="2"
)

{'deleted': 1}

In [52]:
collection1.peek()

{'ids': ['1', '3', '4', '5', '6', '7'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [-0.05396886,  0.04353058, -0.02860478, ...,  0.02393955,
          0.09608028,  0.05749847],
        [-0.04155244, -0.02362224, -0.01487357, ..., -0.03260471,
         -0.0798054 , -0.04597207],
        [ 0.00859756,  0.037237  , -0.00891263, ..., -0.06948347,
         -0.10250813, -0.0632599 ],
        [-0.13393995, -0.00590101, -0.01544957, ...,  0.01599919,
         -0.01226634, -0.05547559],
        [-0.1351217 , -0.0036003 , -0.00462356, ...,  0.01359502,
          0.00168626, -0.05872585]], shape=(6, 384)),
 'documents': ['This is the first document',
  'This is the third document',
  'Hello world 222!',
  'Goodbye World 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'verse': 16, 'chapter': 3},
  {'chapter': 1

In [55]:
collection1.delete(
    ids=["2", "3", "4"]
)

{'deleted': 3}

In [56]:
collection1.peek()

{'ids': ['1', '5', '6', '7'],
 'embeddings': array([[-0.05846755,  0.07448346,  0.02305873, ...,  0.06951981,
          0.1160405 ,  0.05364617],
        [ 0.00859756,  0.037237  , -0.00891263, ..., -0.06948347,
         -0.10250813, -0.0632599 ],
        [-0.13393995, -0.00590101, -0.01544957, ...,  0.01599919,
         -0.01226634, -0.05547559],
        [-0.1351217 , -0.0036003 , -0.00462356, ...,  0.01359502,
          0.00168626, -0.05872585]], shape=(4, 384)),
 'documents': ['This is the first document',
  'Goodbye World 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'verse': 7, 'chapter': 5},
  {'chapter': 12, 'verse': 13},
  {'chapter': 9, 'price': 50.0, 'verse': 6}]}